# Deploy IAM Roles for Tenant Groups

Creates IAM roles for policyholders, adjusters, and administrators groups with Athena/S3 permissions.
These roles are required before setting up Lake Formation permissions on S3 Tables.

These per-group roles are the backbone of the claims path's fine-grained access control: the REQUEST interceptor maps the caller's group claim to one of these roles and assumes it, so **Lake Formation** applies that persona's column filtering and table grants to every Athena query — and the same group→role map in DynamoDB backs the tool-access gating (which tools each group may call).

Note that these roles are per-**group**, not per-user. Per-user *row* scope is a separate mechanism: the claims tools bind the interceptor-propagated caller identity into a `WHERE user_id = ?` predicate. Lake Formation row-level data-cell filters are not configured in this sample.

## Prerequisites

- ✅ AWS credentials configured (via AWS CLI, environment variables, or `.env` file)
- ✅ Python 3.10+ with virtual environment and dependencies installed
- ✅ Run `01-deploy-idp.ipynb` first

## What This Notebook Does

1. Creates IAM roles for tenant groups (policyholders, adjusters, administrators)
2. Attaches Athena/S3 permissions to each role
3. Saves role ARNs to SSM Parameter Store

## Next Notebook

- **03-deploy-s3tables.ipynb**

In [ ]:
# AWS Initialization - Load credentials and create session
from utils.notebook_init import init_aws
import subprocess
import sys

session, region, account_id = init_aws()

# Initialize AWS clients
ssm_client = session.client("ssm", region_name=region)

print("✅ Ready to proceed with AWS operations")
print(f"   Account ID: {account_id}")
print(f"   Region: {region}")

## Step 1: Create Gateway Interceptor Lambda Role

The tenant roles need to trust the Gateway Interceptor Lambda role (so it can assume them during token exchange).
This role must exist before creating the tenant roles.

In [ ]:
import os

# Pass environment so subprocess inherits AWS credentials from init_aws()
env = os.environ.copy()

result = subprocess.run(
    [sys.executable, "create_lambda_role.py"],
    cwd="deployment/5a-gateway-setup",
    capture_output=True,
    text=True,
    env=env,
)

print(result.stdout)
if result.returncode != 0:
    print("❌ Error:", result.stderr)
else:
    print("\n✅ Gateway Interceptor Lambda role created!")
    print("⏳ Waiting for IAM role to propagate...")
    import time

    time.sleep(10)

## Step 2: Deploy IAM Tenant Roles

Each tenant role's trust policy has three statements: the `bedrock.amazonaws.com` service principal, the account root, and — explicitly — the Gateway Interceptor Lambda role created in Step 1, so the interceptor can assume the tenant role during token exchange.

### Why this step retries, and what IAM is actually doing

**IAM's read path and its principal-validation path are separately eventually consistent.** A role that `GetRole` already returns can still be rejected as an invalid principal in another role's trust policy for a few seconds after it was created. So verifying the role exists — which `setup_iam_roles.py` does, and reports as `✅ Verified role exists in IAM` — is a **necessary but not sufficient** precondition for naming it as a principal. `CreateRole` then fails with `MalformedPolicyDocument: Invalid principal in policy`.

This is a genuinely useful thing to know when building on AWS: *"the resource exists"* and *"every service that validates references to this resource knows it exists"* are two different claims, and IAM makes the gap observable. The general rule is to treat cross-resource references as retryable for a short window after a create, rather than assuming a successful read has settled the question.

**Why it shows up here in particular.** Step 1 and Step 2 are adjacent cells. Running the notebook with **"Run All Cells"** executes them seconds apart — on a clean-slate deployment the measured gap was **four seconds**, well inside the window — so Run-All is the reliable way to hit it, not an unlucky edge case. Stepping through the cells by hand is usually slow enough to succeed.

That asymmetry is what makes the failure **intermittent**, and intermittent is worse than deterministic here: a trust-policy error that appears on one run and not the next invites you to doubt your credentials, your account, or your policy JSON — none of which is wrong. So `setup_iam_roles.py` retries `CreateRole` with backoff (2/4/8/16s, ~30s total) and **says so while it waits**, and it retries *only* the invalid-principal form of `MalformedPolicyDocument`. A genuinely malformed policy document raises the same exception class and fails immediately rather than looping.

**What the fix deliberately does not do:** it does not drop the explicit Lambda-role statement, and it does not replace it with the account-root principal. Either would make the error disappear by widening the trust policy — discarding the least-privilege intent that statement exists to express. Waiting for consistency is the correct response to a consistency problem.

In [ ]:
result = subprocess.run(
    [sys.executable, "setup_iam_roles.py"],
    cwd="deployment/2-lakehouse-tenant-roles-setup",
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.returncode != 0:
    print("❌ Error:", result.stderr)
else:
    print("\n✅ IAM tenant roles created!")
    print("\n📋 Role ARNs saved to SSM Parameter Store")

## Step 3: Verify IAM Roles in SSM

In [ ]:
print("Verifying IAM role parameters in SSM...\n")

parameters_to_check = [
    "/app/lakehouse-agent/roles/lakehouse-policyholders-role",
    "/app/lakehouse-agent/roles/lakehouse-adjusters-role",
    "/app/lakehouse-agent/roles/lakehouse-administrators-role",
]

all_found = True
for param_name in parameters_to_check:
    try:
        response = ssm_client.get_parameter(Name=param_name)
        value = response["Parameter"]["Value"]
        print(f"✅ {param_name}")
        print(f"   Value: {value}")
    except ssm_client.exceptions.ParameterNotFound:
        print(f"❌ {param_name} - NOT FOUND")
        all_found = False
    except Exception as e:
        print(f"⚠️  {param_name} - ERROR: {e}")
        all_found = False

if all_found:
    print("\n✅ All IAM role parameters verified in SSM!")
else:
    print("\n⚠️  Some role parameters are missing. Re-run the setup script.")

## Summary

✅ **IAM Tenant Roles Deployment Complete!**

**Roles Created:**
- `lakehouse-policyholders-role`
- `lakehouse-adjusters-role`
- `lakehouse-administrators-role`

**Next Steps:**
Run **03-deploy-s3tables.ipynb** to create the S3 Tables database and tables.